# KPI lab: how good is Thunder-X?

For each benchmark row: send the instruction to Thunder-X, get its XPath back, and check it on the page.
The answer is right when its XPath points to exactly one element, and it's the same element as `expected_xpath`.

Start Thunder-X first: `./run.sh`

In [5]:
import asyncio
import sqlite3
import time

import httpx
from browser_use import BrowserSession

THUNDERX = "http://localhost:8000"
DB = "../data/bench/bench_20261101_231635.sqlite"

## Load the benchmark rows, grouped by page

In [6]:
connection = sqlite3.connect(DB)
connection.row_factory = sqlite3.Row
rows = []
for row in connection.execute("SELECT * FROM samples"):
    rows.append(dict(row))
connection.close()

pages = {}
for row in rows:
    url = row["url"]
    if url not in pages:
        pages[url] = []
    pages[url].append(row)

print(len(rows), "rows on", len(pages), "pages")

178 rows on 11 pages


## Helpers

- `ask_thunderx`: send one instruction, get Thunder-X's answer and how long it took
- `is_right`: check Thunder-X's XPath against the expected one, on the page

In [7]:
#NB :position is the step's place in the session's chain: 1, 2, 3...

In [8]:
http = httpx.Client(base_url=THUNDERX, timeout=120)


def ask_thunderx(session_id, position, instruction):
    start = time.time()
    request = {"session_id": session_id, "position": position, "instruction": instruction, "mode": "validate"}
    answer = http.post("/run", json=request).json()
    seconds = time.time() - start
    return answer, seconds
    
#const found    = document.evaluate(xpath, document, null, 7, null);           // run Thunder-X's XPath
#const expected = document.evaluate(expected_xpath, document, null, 7, null);  // run the expected XPath
# document.evaluate is the browser's own XPath engine. The 7 asks for the list of every element the XPath matches.
# more than 1 ambigious 
# match

SAME_ELEMENT_JS = """(xpath, expected_xpath) => {
  const found = document.evaluate(xpath, document, null, 7, null);
  const expected = document.evaluate(expected_xpath, document, null, 7, null);
  if (found.snapshotLength !== 1) return false;       
  return found.snapshotItem(0) === expected.snapshotItem(0);
}"""

#snapshotItem(0) is the element that was found. === checks whether it's the very


async def is_right(page, xpath, expected_xpath):
    if expected_xpath == "NONE":
        return xpath is None
    if xpath is None:
        return False
    try:
        answer = await page.evaluate(SAME_ELEMENT_JS, xpath, expected_xpath)
    except Exception:
        return False
    return answer == "True"

## Run

For each page: Thunder-X opens it, and the lab opens it too (to check the answers).
Then each row of that page is sent to Thunder-X, one by one.

In [9]:
browser = BrowserSession(headless=True, keep_alive=True)
await browser.start()
results = []

for url in pages:
    print("page:", url)
    session_id = http.post("/sessions", json={"url": url, "title": "kpi lab"}).json()["session_id"]
    await browser.navigate_to(url)
    await asyncio.sleep(2)
    page = await browser.get_current_page()

    position = 1
    for row in pages[url]:
        answer, seconds = ask_thunderx(session_id, position, row["instruction"])
        right = await is_right(page, answer.get("xpath"), row["expected_xpath"])

        result = {
            "instruction": row["instruction"],
            "category": row["category"],
            "expected_action": row["action"],
            "expected_value": row["value"],
            "expected_xpath": row["expected_xpath"],
            "action": answer.get("action"),
            "value": answer.get("value"),
            "xpath": answer.get("xpath"),
            "status": answer.get("status"),
            "route": answer.get("route"),
            "confidence": answer.get("confidence"),
            "seconds": seconds,
            "right": right,
        }
        results.append(result)

        if right:
            print("   right", round(seconds, 1), "s", row["instruction"])
        else:
            print("   WRONG", round(seconds, 1), "s", row["instruction"])
        position = position + 1

    http.delete("/sessions/" + session_id)

await browser.kill()

INFO     [BrowserSession] Setting viewport to 1920x1200 with device scale factor 1.0 whereas original device scale factor was 1.0


page: https://www.selenium.dev/selenium/web/web-form.html
   WRONG 2.8 s Find the text 'Web form'
   WRONG 3.5 s Locate 'Web form'
   right 0.9 s Fill 'Text input' with 'Jane Doe'
   right 1.0 s Enter 'Jane Doe' in 'Text input'
   right 0.1 s Click on 'Submit'
   right 0.2 s Press 'Submit'
   right 2.0 s Double-click on 'File input'
   right 3.1 s Double click the 'File input' button
   right 0.1 s Right-click on 'Return to index'
   right 0.1 s Right click the 'Return to index' link
   right 2.5 s Choose 'One' from 'Dropdown (select)'
   right 2.4 s Select 'One' in the 'Dropdown (select)' dropdown
   WRONG 2.0 s Check 'Checked radio'
   WRONG 2.4 s Tick the 'Checked radio' box
   right 1.8 s Enter 'hello world' in 'Readonly input'
   right 1.9 s Type 'hello world' into the 'Readonly input' field
   WRONG 1.9 s Select the 'Default radio' option
   WRONG 1.8 s Tick the 'Default radio' box
page: https://testpages.eviltester.com/styled/basic-html-form-test.html
   WRONG 1.1 s Click the 'T

INFO     [BrowserSession] 📢 on_BrowserStopEvent - Calling reset() (force=True, keep_alive=True)
INFO     [BrowserSession] [SessionManager] Cleared all owned data (targets, sessions, mappings)
INFO     [BrowserSession] ✅ Browser session reset complete
INFO     [BrowserSession] ✅ Browser session reset complete


## KPIs

- **accuracy**: right answers / all rows
- **coverage**: rows where Thunder-X gave an XPath / all rows
- **precision**: right answers / rows where Thunder-X gave an XPath
- **action accuracy**: Thunder-X read the right action from the instruction
- **value accuracy**: Thunder-X read the right value (only rows that have one)
- **time**: seconds per instruction

In [13]:
def percent(part, whole):
    if whole == 0:
        return "-"
    return str(round(100 * part / whole)) + " %"


total = len(results)
right = 0
answered = 0
right_when_answered = 0
right_action = 0
with_value = 0
right_value = 0
times = []

for r in results:
    times.append(r["seconds"])
    if r["right"]:
        right = right + 1
    if r["xpath"] is not None:
        answered = answered + 1
        if r["right"]:
            right_when_answered = right_when_answered + 1
    if r["action"] == r["expected_action"]:
        right_action = right_action + 1
    if r["expected_value"] is not None:
        with_value = with_value + 1
        if str(r["value"]).lower() == str(r["expected_value"]).lower():
            right_value = right_value + 1

times.sort()
print("rows           ", total)
print("accuracy       ", percent(right, total))
print("coverage       ", percent(answered, total))
print("precision      ", percent(right_when_answered, answered))
print("action accuracy", percent(right_action, total))
print("time average   ", round(sum(times) / total, 2), "s")
print("time median    ", round(times[total // 2], 2), "s")
print("time slowest   ", round(times[-1], 2), "s")

rows            178
accuracy        67 %
coverage        98 %
precision       68 %
action accuracy 78 %
time average    0.78 s
time median     0.54 s
time slowest    3.51 s


## Wrong answers

In [14]:
for r in results:
    if not r["right"]:
        print(r["instruction"])
        print("   expected:", r["expected_xpath"])
        print("   got:     ", r["xpath"], "|", r["status"], "|", r["route"])

Find the text 'Web form'
   expected: //h1[normalize-space()='Web form']
   got:      None | not_found | laya
Locate 'Web form'
   expected: //h1[normalize-space()='Web form']
   got:      //a[normalize-space()='Return to index'] | verified | laya
Check 'Checked radio'
   expected: /html[1]/body[1]/main[1]/div[1]/form[1]/div[1]/div[2]/div[2]/label[1]/input[1]
   got:      //label[normalize-space()='Checked radio'] | verified | laya
Tick the 'Checked radio' box
   expected: /html[1]/body[1]/main[1]/div[1]/form[1]/div[1]/div[2]/div[2]/label[1]/input[1]
   got:      //label[normalize-space()='Checked radio'] | verified | laya
Select the 'Default radio' option
   expected: /html[1]/body[1]/main[1]/div[1]/form[1]/div[1]/div[2]/div[3]/label[1]/input[1]
   got:      //label[normalize-space()='Default radio'] | verified | laya
Tick the 'Default radio' box
   expected: /html[1]/body[1]/main[1]/div[1]/form[1]/div[1]/div[2]/div[3]/label[1]/input[1]
   got:      //label[normalize-space()='Default 

# Generalist LLMs on the same benchmark

An LLM can't open a URL: it only reads the text it's given. So the lab does that part for it:
open the page, take its HTML, clean it (no scripts, styles, drawings; only useful attributes), and send **HTML + instruction**.
The LLM answers `{"xpath", "action", "value"}`, graded with the same `is_right` as Thunder-X.

Same cleaned HTML, same prompt, temperature 0 when the model allows it.

In [ ]:
OPENAI_KEY = ""       # ⚠ paste keys for local runs, REMOVE before pushing (empty → env OPENAI_API_KEY ...)
ANTHROPIC_KEY = ""
GEMINI_KEY = ""

LLMS = [
    {"name": "gpt-5-mini",        "vendor": "openai",    "model": "gpt-5-mini"},
    {"name": "gpt-5",             "vendor": "openai",    "model": "gpt-5"},
    {"name": "claude-haiku-5-5",  "vendor": "anthropic", "model": "claude-haiku-5-5"},
    {"name": "claude-sonnet-5-5", "vendor": "anthropic", "model": "claude-sonnet-5-5"},
    {"name": "gemini-2.5-flash",  "vendor": "gemini",    "model": "gemini-2.5-flash"},
    {"name": "gemini-2.5-pro",    "vendor": "gemini",    "model": "gemini-2.5-pro"},
]
MAX_HTML_CHARS = 150000      # a longer page is cut (the row is marked "cut")
ROWS_PER_PAGE = None         # e.g. 3 for a quick, cheap try

import os

if not OPENAI_KEY:
    OPENAI_KEY = os.environ.get("OPENAI_API_KEY", "")
if not ANTHROPIC_KEY:
    ANTHROPIC_KEY = os.environ.get("ANTHROPIC_API_KEY", "")
if not GEMINI_KEY:
    GEMINI_KEY = os.environ.get("GEMINI_API_KEY", "")
KEYS = {"openai": OPENAI_KEY, "anthropic": ANTHROPIC_KEY, "gemini": GEMINI_KEY}

for llm in LLMS:
    if KEYS[llm["vendor"]]:
        print("ready  ", llm["name"])
    else:
        print("no key ", llm["name"], "(skipped)")

## Helpers

- `clean_html`: the page's HTML without what doesn't help find an element
- `ask_llm`: send the prompt to one LLM, get back its answer, the time and the tokens

In [ ]:
import json
import re

CLEAN_HTML_JS = r"""() => {
  const copy = document.documentElement.cloneNode(true);
  for (const node of copy.querySelectorAll('script, style, noscript, svg, link, meta, template')) node.remove();
  const keep = ['id', 'name', 'class', 'type', 'href', 'role', 'title', 'placeholder', 'aria-label', 'value', 'for', 'alt', 'data-testid'];
  for (const el of copy.querySelectorAll('*')) {
    for (const attribute of [...el.attributes]) {
      if (!keep.includes(attribute.name)) el.removeAttribute(attribute.name);
    }
  }
  return copy.outerHTML.replace(/\s+/g, ' ');
}"""

PROMPT = """You get a web page's HTML and an instruction from a tester.
Find the one element the instruction is about.

Answer with JSON only:
{"xpath": "...", "action": "...", "value": "..."}

- xpath: an XPath that matches exactly that element on the page
- action: one of click, double_click, right_click, hover, type, check, select, read
- value: the text to type or the option to choose, else null
- if the element is not on the page: {"xpath": null, "action": null, "value": null}

Instruction: INSTRUCTION

HTML:
PAGE_HTML"""


def read_json(text):
    text = text.strip()
    text = re.sub(r"^```(json)?", "", text).strip()
    text = re.sub(r"```$", "", text).strip()
    return json.loads(text)


def ask_llm(llm, prompt):
    key = KEYS[llm["vendor"]]
    start = time.time()

    if llm["vendor"] == "openai":
        reply = httpx.post("https://api.openai.com/v1/chat/completions", timeout=300,
                           headers={"Authorization": "Bearer " + key},
                           json={"model": llm["model"], "messages": [{"role": "user", "content": prompt}],
                                 "response_format": {"type": "json_object"}}).json()
        text = reply["choices"][0]["message"]["content"]
        tokens_in = reply["usage"]["prompt_tokens"]
        tokens_out = reply["usage"]["completion_tokens"]

    elif llm["vendor"] == "anthropic":
        reply = httpx.post("https://api.anthropic.com/v1/messages", timeout=300,
                           headers={"x-api-key": key, "anthropic-version": "2023-06-01"},
                           json={"model": llm["model"], "max_tokens": 1024, "temperature": 0,
                                 "messages": [{"role": "user", "content": prompt}]}).json()
        text = reply["content"][0]["text"]
        tokens_in = reply["usage"]["input_tokens"]
        tokens_out = reply["usage"]["output_tokens"]

    else:
        url = "https://generativelanguage.googleapis.com/v1beta/models/" + llm["model"] + ":generateContent"
        reply = httpx.post(url, timeout=300, headers={"x-goog-api-key": key},
                           json={"contents": [{"parts": [{"text": prompt}]}],
                                 "generationConfig": {"temperature": 0, "responseMimeType": "application/json"}}).json()
        text = reply["candidates"][0]["content"]["parts"][0]["text"]
        tokens_in = reply["usageMetadata"]["promptTokenCount"]
        tokens_out = reply["usageMetadata"].get("candidatesTokenCount", 0)

    seconds = time.time() - start
    return read_json(text), seconds, tokens_in, tokens_out

## Run

For each page: open it, clean its HTML once. Then each row goes to each LLM.

In [ ]:
browser = BrowserSession(headless=True, keep_alive=True)
await browser.start()

llm_results = {}
for llm in LLMS:
    llm_results[llm["name"]] = []

for url in pages:
    print("page:", url)
    await browser.navigate_to(url)
    await asyncio.sleep(2)
    page = await browser.get_current_page()
    html = await page.evaluate(CLEAN_HTML_JS)
    cut = len(html) > MAX_HTML_CHARS
    html = html[:MAX_HTML_CHARS]
    print("   html:", len(html), "characters", "(cut)" if cut else "")

    page_rows = pages[url]
    if ROWS_PER_PAGE:
        page_rows = page_rows[:ROWS_PER_PAGE]

    for row in page_rows:
        prompt = PROMPT.replace("INSTRUCTION", row["instruction"]).replace("PAGE_HTML", html)
        for llm in LLMS:
            if not KEYS[llm["vendor"]]:
                continue
            try:
                answer, seconds, tokens_in, tokens_out = ask_llm(llm, prompt)
                error = None
            except Exception as problem:
                answer, seconds, tokens_in, tokens_out = {}, 0, 0, 0
                error = str(problem)[:150]
            right = await is_right(page, answer.get("xpath"), row["expected_xpath"])

            result = {
                "instruction": row["instruction"],
                "category": row["category"],
                "expected_action": row["action"],
                "expected_value": row["value"],
                "expected_xpath": row["expected_xpath"],
                "action": answer.get("action"),
                "value": answer.get("value"),
                "xpath": answer.get("xpath"),
                "seconds": seconds,
                "tokens_in": tokens_in,
                "tokens_out": tokens_out,
                "cut": cut,
                "error": error,
                "right": right,
            }
            llm_results[llm["name"]].append(result)

            if right:
                print("   right", llm["name"], round(seconds, 1), "s", row["instruction"])
            else:
                print("   WRONG", llm["name"], round(seconds, 1), "s", row["instruction"], "|", error or answer.get("xpath"))

await browser.kill()

## Thunder-X vs LLMs

One line per system, same rows, same grader.
(If `ROWS_PER_PAGE` was set, the LLMs ran on fewer rows than Thunder-X: compare the rates, not the counts.)

In [ ]:
def scores(results):
    total = len(results)
    right = 0
    answered = 0
    right_when_answered = 0
    right_action = 0
    tokens = 0
    times = []
    for r in results:
        times.append(r["seconds"])
        tokens = tokens + r.get("tokens_in", 0) + r.get("tokens_out", 0)
        if r["right"]:
            right = right + 1
        if r["xpath"] is not None:
            answered = answered + 1
            if r["right"]:
                right_when_answered = right_when_answered + 1
        if r["action"] == r["expected_action"]:
            right_action = right_action + 1
    times.sort()
    return {
        "rows": total,
        "accuracy": percent(right, total),
        "precision": percent(right_when_answered, answered),
        "coverage": percent(answered, total),
        "action": percent(right_action, total),
        "median time": str(round(times[total // 2], 2)) + " s" if total else "-",
        "tokens per row": round(tokens / total) if total else "-",
    }


print(f"{'system':20} {'rows':>5} {'accuracy':>9} {'precision':>10} {'coverage':>9} {'action':>7} {'median time':>12} {'tokens/row':>11}")
systems = {"Thunder-X": results}
for name in llm_results:
    if llm_results[name]:
        systems[name] = llm_results[name]
for name in systems:
    s = scores(systems[name])
    print(f"{name:20} {s['rows']:>5} {s['accuracy']:>9} {s['precision']:>10} {s['coverage']:>9} {s['action']:>7} {s['median time']:>12} {str(s['tokens per row']):>11}")

## What this benchmark compares

**Thunder-X as a whole system against generalist LLMs.** Not Laya against LLMs.

Thunder-X answers most steps with rules, and calls Laya only when the rules can't decide.
Laya is one part of the system, not the system: its share is visible in the "route" breakdown above (`direct_matcher` vs `laya`).
The question here is: *for this job, does a purpose-built system beat sending the page to a large LLM*, on accuracy, time, cost and repeatability.